# Post-processing: active nematic in an elastic solid

Loads the raw numpy snapshots written by `solver.py` (`output/snapshots/step_*.npz`) and produces:

1. Snapshot plots of displacement magnitude, von Mises stress, and the director field at a few times.
2. An animation of the director field reorienting over time, and a separate animation of the stress field, both on the deforming mesh (nodes warped by displacement).

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FuncAnimation, PillowWriter

OUTPUT_ROOT = "outputs"

# Change only this for the run you want to inspect
OUTDIR = os.path.join(
    OUTPUT_ROOT,
    "test runs",
    "test_legacy_lambda_m1",
)

SNAP_DIR = os.path.join(OUTDIR, "snapshots")

# Mirror the run structure inside figures/
REL_RUN = os.path.relpath(OUTDIR, OUTPUT_ROOT)
FIG_DIR = os.path.join("outputs/figures", REL_RUN)
os.makedirs(FIG_DIR, exist_ok=True)

DEFORM_SCALE = 1.0

NBINS_X = 14
NBINS_Y = 34

print("Reading data from:", OUTDIR)
print("Saving figures to:", FIG_DIR)

In [ ]:
snap_files = sorted(glob.glob(os.path.join(SNAP_DIR, "step_*.npz")))
assert snap_files, f"No snapshots found in {SNAP_DIR} -- run run_simulation.py first"

# connectivity is only stored in the first snapshot (the mesh doesn't change topology)
first = np.load(snap_files[0])
connectivity = first["connectivity"]
coords_ref = first["coords"]

snapshots = []
for f in snap_files:
    d = np.load(f)
    snapshots.append({k: d[k] for k in ["t", "coords", "u", "von_mises", "Qxx", "Qxy", "theta", "S"]})

times = np.array([s["t"] for s in snapshots])
print(f"Loaded {len(snapshots)} snapshots, t in [{times[0]:.3f}, {times[-1]:.3f}]")

In [ ]:
def deformed_coords(snap, scale=DEFORM_SCALE):
    return coords_ref + scale * snap["u"]


def triangulation(snap, scale=DEFORM_SCALE):
    xy = deformed_coords(snap, scale)
    return mtri.Triangulation(xy[:, 0], xy[:, 1], connectivity)


def binned_directors(snap, nbins_x=NBINS_X, nbins_y=NBINS_Y):
    """Coarse-grain the director field onto an nbins_x x nbins_y grid.

    Nematic directors are headless (theta and theta+pi are the same state),
    so they can't be averaged as angles directly -- averaging wraps around
    incorrectly near the +/-pi/2 branch cut. Averaging the Cartesian Q-tensor
    components (Qxx, Qxy) is the correct way to coarse-grain a nematic field:
    it's linear, so head-tail symmetry (Q(theta) == Q(theta+pi)) is baked in
    automatically, and the result is exactly what continuum nematic theory
    calls the coarse-grained order parameter.
    """
    xy = deformed_coords(snap)
    Qxx, Qxy = snap["Qxx"], snap["Qxy"]
    x, y = xy[:, 0], xy[:, 1]

    xedges = np.linspace(x.min(), x.max(), nbins_x + 1)
    yedges = np.linspace(y.min(), y.max(), nbins_y + 1)
    ix = np.clip(np.digitize(x, xedges) - 1, 0, nbins_x - 1)
    iy = np.clip(np.digitize(y, yedges) - 1, 0, nbins_y - 1)
    bin_id = ix * nbins_y + iy
    n_bins = nbins_x * nbins_y

    count = np.bincount(bin_id, minlength=n_bins)
    occupied = count > 0
    qxx_avg = np.bincount(bin_id, weights=Qxx, minlength=n_bins)[occupied] / count[occupied]
    qxy_avg = np.bincount(bin_id, weights=Qxy, minlength=n_bins)[occupied] / count[occupied]
    x_c = np.bincount(bin_id, weights=x, minlength=n_bins)[occupied] / count[occupied]
    y_c = np.bincount(bin_id, weights=y, minlength=n_bins)[occupied] / count[occupied]

    theta = 0.5 * np.arctan2(qxy_avg, qxx_avg)
    S = np.sqrt(qxx_avg**2 + qxy_avg**2)
    return x_c, y_c, theta, S


def director_segments(snap, length=None):
    """Line-segment endpoints for a headless director glyph, one per bin."""
    x_c, y_c, theta, S = binned_directors(snap)
    if length is None:
        # a reasonable glyph length relative to the bin spacing
        length = 0.4 * np.ptp(coords_ref[:, 1]) / NBINS_Y
    dx = length * S * np.cos(theta)
    dy = length * S * np.sin(theta)
    x0, y0 = x_c - dx, y_c - dy
    x1, y1 = x_c + dx, y_c + dy
    return x0, y0, x1, y1


## Snapshot plots at a few times

In [ ]:
sample_idx = np.linspace(0, len(snapshots) - 1, 4).astype(int)

fig, axes = plt.subplots(1, len(sample_idx), figsize=(4 * len(sample_idx), 5), sharey=True)
for ax, i in zip(axes, sample_idx):
    snap = snapshots[i]
    tri = triangulation(snap)
    umag = np.linalg.norm(snap["u"], axis=1)
    im = ax.tricontourf(tri, umag, levels=20, cmap="viridis")
    ax.triplot(tri, linewidth=0.1, color="k", alpha=0.2)
    ax.set_title(f"t = {snap['t']:.2f}")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes, label="|u|", shrink=0.8)
fig.suptitle("Displacement magnitude")
fig.savefig(
    os.path.join(FIG_DIR, "displacement_snapshots.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(sample_idx), figsize=(4 * len(sample_idx), 5), sharey=True)
for ax, i in zip(axes, sample_idx):
    snap = snapshots[i]
    tri = triangulation(snap)
    im = ax.tricontourf(tri, snap["von_mises"], levels=20, cmap="inferno")
    ax.triplot(tri, linewidth=0.1, color="k", alpha=0.2)
    ax.set_title(f"t = {snap['t']:.2f}")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes, label="von Mises stress", shrink=0.8)
fig.suptitle("Stress field")
fig.savefig(
    os.path.join(FIG_DIR, "stress_snapshots.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(sample_idx), figsize=(4 * len(sample_idx), 5), sharey=True)
for ax, i in zip(axes, sample_idx):
    snap = snapshots[i]
    tri = triangulation(snap)
    ax.tricontourf(tri, snap["S"], levels=20, cmap="Greys_r", alpha=0.6)
    x0, y0, x1, y1 = director_segments(snap)
    ax.plot(np.vstack([x0, x1]), np.vstack([y0, y1]), color="crimson", linewidth=1.0)
    ax.set_title(f"t = {snap['t']:.2f}")
    ax.set_aspect("equal")
fig.suptitle("Director field (background: nematic order S)")
fig.savefig(
    os.path.join(FIG_DIR, "director_snapshots.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Animations

Two movies on the deforming mesh: the director field reorienting, and the stress field.

In [ ]:
def make_director_animation(path=None, fps=10):
    path = path or os.path.join(FIG_DIR, "director_animation.gif")
    fig, ax = plt.subplots(figsize=(6, 8))

    def draw(i):
        ax.clear()
        snap = snapshots[i]
        tri = triangulation(snap)
        ax.tricontourf(tri, snap["S"], levels=20, cmap="Greys_r", alpha=0.6, vmin=0, vmax=1.05)
        x0, y0, x1, y1 = director_segments(snap)
        ax.plot(np.vstack([x0, x1]), np.vstack([y0, y1]), color="crimson", linewidth=1.0)
        ax.set_aspect("equal")
        ax.set_title(f"director field, t = {snap['t']:.2f}")
        return []

    anim = FuncAnimation(fig, draw, frames=len(snapshots), blit=False)
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return path


def make_stress_animation(path=None, fps=10):
    path = path or os.path.join(FIG_DIR, "stress_animation.gif")
    fig, ax = plt.subplots(figsize=(6, 8))
    vmax = max(s["von_mises"].max() for s in snapshots)

    def draw(i):
        ax.clear()
        snap = snapshots[i]
        tri = triangulation(snap)
        ax.tricontourf(tri, snap["von_mises"], levels=20, cmap="inferno", vmin=0, vmax=vmax)
        ax.set_aspect("equal")
        ax.set_title(f"von Mises stress, t = {snap['t']:.2f}")
        return []

    anim = FuncAnimation(fig, draw, frames=len(snapshots), blit=False)
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return path

In [ ]:
director_path = make_director_animation()
stress_path = make_stress_animation()
print(director_path, stress_path)

In [ ]:
from IPython.display import Image
Image(filename=director_path)

In [ ]:
Image(filename=stress_path)